# CNN Sequence Model (PyTorch)

Notebook version of `cnn_model.py`, split into runnable blocks.

- Uses sequence one-hot inputs (`clean_data/train` and `clean_data/test`)
- Uses 10% of test as in-memory validation for early stopping
- Uses Mac GPU (`mps`) when available
- Prints final test accuracy

In [1]:
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [2]:
torch.manual_seed(42)
np.random.seed(42)

BASE = Path('.')
CLEAN = BASE / 'clean_data'

In [3]:
class SequenceDataset(Dataset):
    def __init__(self, sequence_array: np.ndarray, labels: np.ndarray):
        self.sequence_array = sequence_array
        self.labels = labels.astype(np.int64)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        seq = self.sequence_array[idx]  # shape: (L, 4)
        y = self.labels[idx]
        return seq, y


def collate_sequences(batch):
    sequences, labels = zip(*batch)
    max_len = max(seq.shape[0] for seq in sequences)
    batch_size = len(sequences)

    x = torch.zeros((batch_size, 4, max_len), dtype=torch.float32)
    for i, seq in enumerate(sequences):
        seq_tensor = torch.from_numpy(seq).float().T  # (4, L)
        x[i, :, : seq_tensor.shape[1]] = seq_tensor

    y = torch.tensor(labels, dtype=torch.long)
    return x, y

In [8]:
class DNAConvNet(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        pooled_len = 10
        self.features = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(pooled_len),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256 * pooled_len, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def pick_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def stratified_split_indices(
    labels: np.ndarray, val_fraction: float = 0.10, seed: int = 42
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)

    val_indices = []
    test_indices = []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)
        n_val = max(1, int(round(len(cls_idx) * val_fraction)))
        val_indices.append(cls_idx[:n_val])
        test_indices.append(cls_idx[n_val:])

    val_indices = np.concatenate(val_indices)
    test_indices = np.concatenate(test_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return val_indices, test_indices


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

In [9]:
train_seq = np.load(CLEAN / 'train' / 'sequence_onehot.npy', allow_pickle=True)
train_targets = pd.read_csv(CLEAN / 'train' / 'targets.csv')['class_index'].to_numpy()

test_seq_full = np.load(CLEAN / 'test' / 'sequence_onehot.npy', allow_pickle=True)
test_targets_full = pd.read_csv(CLEAN / 'test' / 'targets.csv')['class_index'].to_numpy()

if len(train_seq) != len(train_targets):
    raise ValueError('Train sequence/target length mismatch.')
if len(test_seq_full) != len(test_targets_full):
    raise ValueError('Test sequence/target length mismatch.')

val_idx, final_test_idx = stratified_split_indices(test_targets_full, val_fraction=0.10, seed=42)
val_seq = test_seq_full[val_idx]
val_targets = test_targets_full[val_idx]
test_seq = test_seq_full[final_test_idx]
test_targets = test_targets_full[final_test_idx]

num_classes = int(max(train_targets.max(), test_targets_full.max()) + 1)

train_ds = SequenceDataset(train_seq, train_targets)
val_ds = SequenceDataset(val_seq, val_targets)
test_ds = SequenceDataset(test_seq, test_targets)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_sequences)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_sequences)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_sequences)

device = pick_device()
print(f'Using device: {device}')
print(f'Train size: {len(train_ds)}')
print(f'Validation size (10% of original test): {len(val_ds)}')
print(f'Final test size: {len(test_ds)}')

Using device: mps
Train size: 19023
Validation size (10% of original test): 477
Final test size: 4279


In [10]:
model = DNAConvNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

max_epochs = 30
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_state = None

for epoch in range(1, max_epochs + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * yb.size(0)
        seen += yb.size(0)

    train_loss = running_loss / seen
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(
        f"Epoch {epoch:02d}/{max_epochs} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping triggered at epoch {epoch}.')
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

RuntimeError: linear(): input and weight.T shapes cannot be multiplied (64x2560 and 256x128)

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Final Test Loss: {test_loss:.4f}')
print(f'Final Test Accuracy: {test_acc:.4f}')

Final Test Loss: 0.4620
Final Test Accuracy: 0.8406
